In [37]:
import numpy as np

In [38]:
# circle: a boolean variable (type bool), indicating if the player must land exactly on
# the final, goal, square 15 to win (circle = True) or still wins by overstepping the final
# square (circle = False).
circle = True

# layout: a vector of type numpy.ndarray that represents the layout of the game, containing 15 values
#         representing the 15 squares of the Snakes and Ladders game:
# layout[i] = 0 if it is an ordinary square
#           = 1 if it is a “restart” trap (go back to square 1)
#           = 2 if it is a “penalty” trap (go back 3 steps)
#           = 3 if it is a “prison” trap (skip next turn)
#           = 4 if it is a “mystery” trap (random effect among the three previous)
# Note that the first and final squares cannot be trapped.

np.random.seed(42)
layout = np.random.choice([1, 2, 3, 4], 15)
layout[0] = 0
layout[14] = 0
print (layout)

[0 4 1 3 3 4 1 1 3 2 3 3 3 3 0]


In [39]:
from part_I import  value_iteration
nb_turn_solo = value_iteration(layout, circle, 0.001, alpha=1)
nb_turn_solo = nb_turn_solo[0]

In [40]:

def expected_new_places(state, action, layout, circle):

    def rolls_security_dice():
        possible_trap_triggered = [False]
        possible_dice_rolls = [0, 1]
        p = 1/2
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    def rolls_normal_dice():
        possible_trap_triggered = [True, False]
        possible_dice_rolls = [0, 1, 2]
        p = 1/6
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    


    def rolls_risky_dice():
        possible_trap_triggered = [True]
        possible_dice_rolls = [0, 1, 2, 3]
        p = 1/4
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    rolls_dice_functions = {0:rolls_security_dice, 1:rolls_normal_dice, 2:rolls_risky_dice} 



    (current_position, current_skip_next_turn) = state

    
    if current_skip_next_turn:
        new_state = (current_position, False)
        return [(1, new_state)]
    

    # roll dices
    roll_function = rolls_dice_functions[action]
    list_rolls =  roll_function()

    list_new_positions_before_traps = []

    for (p, trap_triggered, dice_roll) in list_rolls:
        # find new position
        if dice_roll==0:
            new_position_before_trap = current_position
            list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))
        else: # dice roll is not 0
            if (current_position == 2):
                    new_position_before_trap = current_position + dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))
                    new_position_before_trap = 9+dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))


            elif  current_position in range(10): # but not 2
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 10:
                    if circle:
                        new_position_before_trap -= 10
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

            elif current_position in range(10, 14):
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 14:
                    if circle:
                        new_position_before_trap -=14
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

    list_new_places_after_traps = []

    for (p, new_position_before_trap, trap_triggered) in list_new_positions_before_traps:
        # Deal with the traps
        trap = layout[new_position_before_trap]

        if  (not trap_triggered) or (trap == 0):
            new_skip_next_turn = False
            new_position_after_trap = new_position_before_trap
            list_new_places_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))
        else:
            trap_list = []

            if trap == 4:
                trap_list.append(1)
                trap_list.append(2)
                trap_list.append(3)
                p/=3
            else:
                trap_list.append(trap)
            
            for trap in trap_list:
                if   trap == 1:
                    new_position_after_trap = 0
                    new_skip_next_turn = False

                elif trap == 2:
                    new_position_after_trap = new_position_before_trap
                    if new_position_after_trap in range(10, 13):
                        new_position_after_trap -= 7 # -7 -3 = -10
                    new_position_after_trap = max(0, new_position_after_trap - 3)
                    new_skip_next_turn = False


                elif trap == 3:
                    new_position_after_trap = new_position_before_trap
                    new_skip_next_turn=True

                list_new_places_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))

    return list_new_places_after_traps




def expected_probability_win(state, action, P, layout, circle, nb_turn):
    # state = (position1:(), positions2:(), whoseturn)
    # probabilité de gagner du joueur 1
    # if whoseturn==1:-> max
    # if whoseturn==2:-> min

    (my_place, other_place, whoseturn) = state

    if other_place[0] == 14:
        if whoseturn==2: return 1, np.array([0., nb_turn_solo[my_place]])
        else: return 0, np.array([0., nb_turn_solo[my_place]])
    
    list_expected_my_new_places = expected_new_places(my_place, action, layout, circle)

    new_probability  = 0
    new_nb_turn = np.array([1., 0.])
    
    for (p, new_place) in list_expected_my_new_places:
        new_probability += p*P[(other_place, new_place, (whoseturn-1)*1 + (2-whoseturn)*2)] # (whoseturn-1)*1 + (2-whoseturn)*2) -> if whoseturn is 1 return 2, if whoseturn is 2 return 1
        b = nb_turn[(other_place, new_place, (whoseturn-1)*1 + (2-whoseturn)*2)]
        new_nb_turn +=p*np.array([b[1], b[0]])

    return new_probability, new_nb_turn

    
    
def min_max(layout, circle, theta):

    # probabilité de gagner du joueur 1
    # if whoseturn==1:-> max
    # if whoseturn==2:-> min
    
    possible_places = []
    for i in range(14):
        if layout[i]>=3:
            possible_places.append((i, False))
            possible_places.append((i, True))
        else:
            possible_places.append((i, False))

    # (my_place, other_place, whoseturn) = state
    possible_states = []
    for place1 in possible_places:
        for place2 in possible_places:
            possible_states.append((place1, place2, 1))
            possible_states.append((place1, place2, 2))

        possible_states.append((place1, (14, False), 1))
        possible_states.append((place1, (14, False), 2))

    P = {} # probability 1 win
    best_policy = {}
    nb_turn = {}


    for state in possible_states:
        P[state] = 0
        best_policy[state] = 0
        nb_turn[state] = np.array([0., 0.])
    
    delta = 2*theta
    while delta>=theta:
        print(delta)
        delta = 0
        for state in possible_states:
            v = P[state]
            (_, _, whoseturn) = state

            if whoseturn == 2:
                minv = np.inf
                for action in range(3):
                    Pn, new_nb_turn = expected_probability_win(state, action, P, layout, circle, nb_turn)
                    # print(nv)
                    if Pn <= minv:
                        minv = Pn
                        best_policy[state] = action
                        nb_turn[state] = new_nb_turn

                P[state] = minv
            else :  # whoseturn == 1
                maxv = -np.inf
                for action in range(3):
                    Pn, new_nb_turn = expected_probability_win(state, action, P, layout, circle, nb_turn)
                    if Pn >= maxv:
                        maxv = Pn
                        best_policy[state] = action
                        nb_turn[state] = new_nb_turn
                P[state] = maxv
            delta = max(abs(v-P[state]), delta)
    print(delta)
    return P, best_policy, nb_turn
        


In [41]:


P, best_policy, nb_turn = min_max(layout, circle, 0.000000001)

# 1512

#  ((8, True), (13, False), 2): 2,
#  ((9, False), (13, False), 1): 0,

2e-09
1
0.5
0.49999999999999994
0.28125
0.22761140046296302
0.17138350931105678
0.16901909858662134
0.12204041248243663
0.11014387891424793
0.09252140979829504
0.08836845793474646
0.08114973796721792
0.07128636300067104
0.06034968367941684
0.04831451349047766
0.04544470194666894
0.03784544892288455
0.032915519528883386
0.028557976564113907
0.02468816609326885
0.021364438816494025
0.018002125794116863
0.014880342397548874
0.012021802234603585
0.009506142369018922
0.007382917402407707
0.005715828555390545
0.004739027019874498
0.003922311864330008
0.0032482598144223385
0.0026590044718222128
0.002148918848131176
0.0017150855524559194
0.0013545540612073248
0.0010611740380237489
0.0008270979024187497
0.0006428138924829963
0.0004989852167203157
0.0003872693334119326
0.0003004508506417869
0.00023304603143836644
0.00018073950838670694
0.00014011701368799034
0.00010850537167150787
8.38673667142853e-05
6.467099163853796e-05
4.975093756776072e-05
3.819830917761724e-05
2.9288845512565054e-05
2.2440

In [42]:
nb_turn

{((0, False), (0, False), 1): array([19.14743715, 18.12394652]),
 ((0, False), (0, False), 2): array([19.1474367, 18.123947 ]),
 ((0, False), (1, False), 1): array([17.25323056, 18.17550543]),
 ((0, False), (1, False), 2): array([17.25323024, 18.17550574]),
 ((0, False), (1, True), 1): array([18.28105235, 18.15617391]),
 ((0, False), (1, True), 2): array([18.28105199, 18.15617426]),
 ((0, False), (2, False), 1): array([15.18949773, 18.20258634]),
 ((0, False), (2, False), 2): array([15.18949753, 18.20258655]),
 ((0, False), (3, False), 1): array([20.44847857, 18.12823485]),
 ((0, False), (3, False), 2): array([20.44847827, 18.12823516]),
 ((0, False), (3, True), 1): array([21.50805867, 18.08705564]),
 ((0, False), (3, True), 2): array([21.50805833, 18.08705599]),
 ((0, False), (4, False), 1): array([19.09757509, 18.18569512]),
 ((0, False), (4, False), 2): array([19.09757482, 18.18569539]),
 ((0, False), (4, True), 1): array([20.19739057, 18.15224232]),
 ((0, False), (4, True), 2): arr

In [43]:
nb_turn_solo

{(0, False): 18.10634250536987,
 (1, False): 16.250424126282653,
 (1, True): 17.250424126282653,
 (2, False): 14.25081177401095,
 (3, False): 18.83527083660027,
 (3, True): 19.83527083660027,
 (4, False): 17.666665109743565,
 (4, True): 18.666665109743565,
 (5, False): 15.666666609685581,
 (5, True): 16.66666660968558,
 (6, False): 13.666666660637478,
 (7, False): 11.66666666616656,
 (8, False): 9.666666666636512,
 (8, True): 10.666666666636512,
 (9, False): 7.6666666666655034,
 (10, False): 5.666666666666645,
 (10, True): 6.666666666666645,
 (11, False): 3.9999999999999893,
 (11, True): 4.999999999999989,
 (12, False): 2.999999999999993,
 (12, True): 3.999999999999993,
 (13, False): 1.9999999999999964,
 (13, True): 2.9999999999999964,
 (14, False): 0}